# 12 — DoWhy refutacje (T10)


In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import warnings
warnings.filterwarnings("ignore")

import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, str(Path("..").resolve()))
from src.dataset import OpenBanditDataset
from src.propensity import OBD_COMMON_CONTEXT_DIM

import dowhy
from dowhy import CausalModel

sns.set_theme(style="whitegrid")
np.random.seed(42)

N_FEATURES   = OBD_COMMON_CONTEXT_DIM  # 20
RANDOM_STATE = 42
N_REFUTATIONS = 50  # liczba refutacji per test

FIGURES_DIR = Path("../figures/week10")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
print(f"DoWhy version: {dowhy.__version__}")


## Dane

- T: akcja 0 vs reszta
- Y: reward
- X: 20 cech kontekstu


In [ ]:
ds_bts = OpenBanditDataset(behavior_policy="bts", campaign="all")
fb_bts = ds_bts.obtain_batch_bandit_feedback()

ctx = fb_bts["context"][:, :N_FEATURES]
act = fb_bts["action"]
rew = fb_bts["reward"].astype(np.float64)

treatment = (act == 0).astype(int)

ctx_cols = [f"ctx_{i}" for i in range(N_FEATURES)]
df = pd.DataFrame(ctx, columns=ctx_cols)
df["treatment"] = treatment
df["reward"]    = rew

print(f"Dataset shape: {df.shape}")
print(f"Treatment=1: {treatment.mean():.2%}  Treatment=0: {(1-treatment).mean():.2%}")
print(f"CTR treated:   {rew[treatment==1].mean():.4f}")
print(f"CTR untreated: {rew[treatment==0].mean():.4f}")
print(f"Naive ATE:     {rew[treatment==1].mean() - rew[treatment==0].mean():.4f}")


## Definicja modelu kauzalnego w DoWhy


In [ ]:
common_causes = ctx_cols

model = CausalModel(
    data=df,
    treatment="treatment",
    outcome="reward",
    common_causes=common_causes,
)

print("Model kauzalny zdefiniowany.")
print(f"  Treatment:     treatment")
print(f"  Outcome:       reward")
print(f"  Common causes: {len(common_causes)} cech kontekstu (ctx_0 ... ctx_19)")


## Identyfikacja i estymacja efektu kauzalnego


In [ ]:
identified_estimand = model.identify_effect(proceed_when_unidentifiable=True)
estimate = model.estimate_effect(
    identified_estimand,
    method_name="backdoor.linear_regression",
    control_value=0,
    treatment_value=1,
)

ate = estimate.value
print(f"\nATE (backdoor linear regression): {ate:.6f}")


## Random common cause


In [ ]:
refute_rcc = model.refute_estimate(
    identified_estimand,
    estimate,
    method_name="random_common_cause",
    num_simulations=N_REFUTATIONS,
)

print(refute_rcc)
print(f"\nOriginal ATE:  {ate:.6f}")
print(f"Refuted ATE:   {refute_rcc.new_effect:.6f}")
print(f"p-value:       {refute_rcc.refutation_result.get('p_value', 'N/A')}")


## Placebo treatment


In [ ]:
refute_placebo = model.refute_estimate(
    identified_estimand,
    estimate,
    method_name="placebo_treatment_refuter",
    placebo_type="permute",
    num_simulations=N_REFUTATIONS,
)

print(refute_placebo)
print(f"\nOriginal ATE:  {ate:.6f}")
print(f"Placebo ATE:   {refute_placebo.new_effect:.6f}  (oczekiwane: ≈ 0)")
print(f"p-value:       {refute_placebo.refutation_result.get('p_value', 'N/A')}")


## Data subset (80%)


In [ ]:
refute_subset = model.refute_estimate(
    identified_estimand,
    estimate,
    method_name="data_subset_refuter",
    subset_fraction=0.8,
    num_simulations=N_REFUTATIONS,
)

print(refute_subset)
print(f"\nOriginal ATE:  {ate:.6f}")
print(f"Subset ATE:    {refute_subset.new_effect:.6f}")
print(f"p-value:       {refute_subset.refutation_result.get('p_value', 'N/A')}")


## Wizualizacja wyników refutacji


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

tests = ["Original ATE", "Random Common Cause", "Placebo Treatment", "Data Subset (80%)"]
values = [
    ate,
    refute_rcc.new_effect,
    refute_placebo.new_effect,
    refute_subset.new_effect,
]
colors_r = ["steelblue", "darkorange", "firebrick", "seagreen"]

bars = ax.barh(tests, values, color=colors_r, alpha=0.85)
ax.axvline(0, color="gray", linestyle=":", alpha=0.7)
ax.axvline(ate, color="steelblue", linestyle="--", alpha=0.6, label=f"Original ATE = {ate:.4f}")

for bar, val in zip(bars, values):
    ax.text(val + (0.0001 if val >= 0 else -0.0001),
            bar.get_y() + bar.get_height()/2,
            f"{val:.4f}", va="center", ha="left" if val >= 0 else "right", fontsize=9)

ax.set_xlabel("Estimated ATE")
ax.set_title("Week 10: Sensitivity Analysis — Refutation Tests (DoWhy)")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "sensitivity_refutations.png", dpi=160)
plt.show()
